# SmartMeter AI: Exploratory Data Analysis & Machine Learning Modeling
## Electricity Demand Forecasting & Intelligent Anomaly Detection

This notebook demonstrates the end-to-end data science pipeline:
1. Data loading and data quality audit
2. Time-series preprocessing and missing value imputation
3. Exploratory Data Analysis (EDA) of 876,000 smart meter readings
4. Feature engineering (lag features, rolling statistics, calendar indicators)
5. Chronological splitting (70% Train, 15% Val, 15% Test)
6. Forecasting Model Training: Naive Baseline, Moving Average, and Random Forest Regressor
7. Multi-step recursive 24-hour demand forecasting
8. Unsupervised Anomaly Detection using Isolation Forest
9. Model evaluation and operational anomaly classification

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.append('..')

from src.data_loader import load_raw_data, get_data_summary
from src.preprocessing import clean_smart_meter_data
from src.feature_engineering import engineer_meter_features, get_forecasting_feature_names, get_anomaly_feature_names
from src.forecasting import split_time_series_chronological, predict_naive_baseline, predict_moving_average_baseline, forecast_next_24h_recursive
from src.evaluation import calculate_forecasting_metrics, calculate_anomaly_metrics
from src.anomaly_detection import train_isolation_forest, detect_anomalies

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Data Ingestion & Quality Audit

In [ ]:
raw_path = '../data/smart_meter_data.csv' if os.path.exists('../data/smart_meter_data.csv') else 'data/smart_meter_data.csv'
df_raw = load_raw_data(raw_path)
summary = get_data_summary(df_raw)

print('Dataset Summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

df_raw.head()

## 2. Preprocessing & Missing Value Imputation
Imputing missing consumption values via linear interpolation along each meter's time series.

In [ ]:
df_clean, clean_stats = clean_smart_meter_data(df_raw, save_path=None)
print('Data Cleaning Audit:')
for k, v in clean_stats.items():
    print(f'  {k}: {v}')

## 3. Exploratory Data Analysis (EDA)
Investigating diurnal patterns, seasonal effects, and temperature correlations.

In [ ]:
# Hourly Consumption Profile (Diurnal Cycle)
hourly_profile = df_clean.groupby('hour')['consumption_kwh'].agg(['mean', 'median', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly_profile['hour'], hourly_profile['mean'], marker='o', color='#00d2be', label='Mean Consumption (kWh)', lw=2.5)
ax.plot(hourly_profile['hour'], hourly_profile['median'], marker='s', color='#38bdf8', linestyle='--', label='Median Consumption (kWh)')
ax.set_title('Grid Diurnal Consumption Profile (24-Hour Cycle)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day (0-23)')
ax.set_ylabel('Consumption (kWh)')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Monthly Demand Volume
monthly = df_clean.groupby('month')['consumption_kwh'].sum() / 1000.0

fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(kind='bar', color='#10b981', ax=ax, edgecolor='black')
ax.set_title('Monthly Electricity Consumption Across 100 Smart Meters (MWh)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month (1-12)')
ax.set_ylabel('Total Consumption (MWh)')
plt.tight_layout()
plt.show()

## 4. Feature Engineering
Generating lag features (t-1, t-2, t-24, t-48, t-168) and rolling statistics (mean, std, max, min) on `shift(1)` to strictly prevent target leakage.

In [ ]:
df_features = engineer_meter_features(df_clean, drop_na_lags=True)
print('Engineered DataFrame Shape:', df_features.shape)
df_features[['timestamp', 'meter_id', 'consumption_kwh', 'lag_1', 'lag_24', 'rolling_mean_24', 'rolling_std_24']].head()

## 5. Chronological Train / Validation / Test Split
Strict chronological partitioning (70% train, 15% validation, 15% test) without shuffling.

In [ ]:
train_df, val_df, test_df, split_info = split_time_series_chronological(df_features)
for k, v in split_info.items():
    print(f'  {k}: {v}')

forecasting_cols = get_forecasting_feature_names()
y_test = test_df['consumption_kwh'].values

## 6. Model Training & Benchmarking
Comparing Model 1 (Naive Baseline), Model 2 (Moving Average), and Model 3 (Random Forest Regressor).

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Model 1: Naive
pred_naive = predict_naive_baseline(test_df)
m_naive = calculate_forecasting_metrics(y_test, pred_naive)

# Model 2: Moving Average
pred_ma = predict_moving_average_baseline(test_df)
m_ma = calculate_forecasting_metrics(y_test, pred_ma)

# Model 3: Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=12, max_samples=0.5, random_state=42, n_jobs=-1)
rf.fit(train_df[forecasting_cols].values, train_df['consumption_kwh'].values)
pred_rf = rf.predict(test_df[forecasting_cols].values)
m_rf = calculate_forecasting_metrics(y_test, pred_rf)

comparison = pd.DataFrame([
    {'Model': 'Naive Baseline (lag_24)', **m_naive},
    {'Model': 'Moving Average Baseline (24h)', **m_ma},
    {'Model': 'Random Forest Regressor (Trained)', **m_rf}
])
comparison

## 7. Recursive Multi-Step 24-Hour Forecasting Demo
Simulating real-world forward projections by updating lag features iteratively with prior predictions.

In [ ]:
demo_meter = 'METER_001'
fc_24h = forecast_next_24h_recursive(
    model=rf,
    history_df=df_features,
    meter_id=demo_meter,
    feature_cols=forecasting_cols,
    hours_ahead=24
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(fc_24h['forecast_step'], fc_24h['predicted_kwh'], marker='o', color='#00d2be', label='24h Forecast (Recursive)', lw=2.5)
ax.fill_between(fc_24h['forecast_step'], fc_24h['lower_bound'], fc_24h['upper_bound'], color='#00d2be', alpha=0.2, label='95% Confidence Interval')
ax.set_title(f'Recursive 24-Hour Electricity Demand Forecast for {demo_meter}', fontsize=14, fontweight='bold')
ax.set_xlabel('Forecast Horizon (Hours Ahead)')
ax.set_ylabel('Consumption (kWh)')
ax.set_xticks(range(1, 25))
ax.legend()
plt.tight_layout()
plt.show()

## 8. Unsupervised Anomaly Detection (Isolation Forest)
Fitting Isolation Forest on consumption and rolling deviation features (strictly excluding `is_anomaly`).

In [ ]:
anom_cols = get_anomaly_feature_names()
iso = train_isolation_forest(train_df, anom_cols, contamination=0.025, random_state=42)
scored = detect_anomalies(iso, test_df, anom_cols)

iso_eval = calculate_anomaly_metrics(test_df['is_anomaly'].values, scored['anomaly_flag'].values)
print('Isolation Forest Test Evaluation vs Ground Truth:')
for k, v in iso_eval.items():
    print(f'  {k}: {v}')

## 9. Conclusion & Deployment
The trained Random Forest and Isolation Forest models have been serialized to `models/` and can now be launched within the interactive Streamlit dashboard (`streamlit run app.py`).